In [3]:
import os
os.environ.setdefault("JAVA_HOME", "/usr/lib/jvm/java-17-openjdk-amd64")
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ.get("PATH", "")


In [4]:
from pathlib import Path

PROJECT_ROOT = Path("/home/belal/OmniLake-DEPI-Graduation-Project-")
bronze_path = str(PROJECT_ROOT / "data_lake" / "bronze" / "BTCUSDT")

In [8]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DecimalType, BooleanType
)

spark = (
    SparkSession.builder
    .appName("OmniLake-Silver-BTCUSDT")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

PROJECT_ROOT = Path("/home/belal/OmniLake-DEPI-Graduation-Project-")
bronze_path = str(PROJECT_ROOT / "data_lake" / "bronze" / "BTCUSDT")

raw_schema = StructType([
    StructField("trade_id", LongType(), True),
    StructField("price", DecimalType(18, 8), True),
    StructField("quantity", DecimalType(18, 8), True),
    StructField("quote_quantity", DecimalType(18, 8), True),
    StructField("timestamp_ms", LongType(), True),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True),
])

df_raw = (
    spark.read
    .option("header", "false")
    .schema(raw_schema)
    .csv(bronze_path)
)

df_raw.printSchema()
df_raw.show(5, truncate=False)

root
 |-- trade_id: long (nullable = true)
 |-- price: decimal(18,8) (nullable = true)
 |-- quantity: decimal(18,8) (nullable = true)
 |-- quote_quantity: decimal(18,8) (nullable = true)
 |-- timestamp_ms: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)

+----------+--------------+----------+--------------+-------------+--------------+-------------+
|trade_id  |price         |quantity  |quote_quantity|timestamp_ms |is_buyer_maker|is_best_match|
+----------+--------------+----------+--------------+-------------+--------------+-------------+
|3344774379|42283.58000000|0.00069000|29.17567020   |1704067200000|true          |true         |
|3344774380|42283.59000000|0.00144000|60.88836960   |1704067200001|false         |true         |
|3344774381|42283.58000000|0.00102000|43.12925160   |1704067200003|true          |true         |
|3344774382|42283.59000000|0.00219000|92.60106210   |1704067200003|false         |true         

In [8]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DecimalType, BooleanType
)

spark = (
    SparkSession.builder
    .appName("OmniLake-Silver-BTCUSDT")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

PROJECT_ROOT = Path("/home/belal/OmniLake-DEPI-Graduation-Project-")
bronze_path = str(PROJECT_ROOT / "data_lake" / "bronze" / "BTCUSDT")

raw_schema = StructType([
    StructField("trade_id", LongType(), True),
    StructField("price", DecimalType(18, 8), True),
    StructField("quantity", DecimalType(18, 8), True),
    StructField("quote_quantity", DecimalType(18, 8), True),
    StructField("timestamp_ms", LongType(), True),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True),
])

df_raw = (
    spark.read
    .option("header", "false")
    .schema(raw_schema)
    .csv(bronze_path)
)

df_raw.printSchema()
df_raw.show(5, truncate=False)

root
 |-- trade_id: long (nullable = true)
 |-- price: decimal(18,8) (nullable = true)
 |-- quantity: decimal(18,8) (nullable = true)
 |-- quote_quantity: decimal(18,8) (nullable = true)
 |-- timestamp_ms: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)

+----------+--------------+----------+--------------+-------------+--------------+-------------+
|trade_id  |price         |quantity  |quote_quantity|timestamp_ms |is_buyer_maker|is_best_match|
+----------+--------------+----------+--------------+-------------+--------------+-------------+
|3344774379|42283.58000000|0.00069000|29.17567020   |1704067200000|true          |true         |
|3344774380|42283.59000000|0.00144000|60.88836960   |1704067200001|false         |true         |
|3344774381|42283.58000000|0.00102000|43.12925160   |1704067200003|true          |true         |
|3344774382|42283.59000000|0.00219000|92.60106210   |1704067200003|false         |true         

In [9]:
from pyspark.sql import functions as F

print("Total raw rows:", df_raw.count())
df_raw.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_raw.columns]
).show(truncate=False)

Total raw rows: 52549865


[Stage 4:=====================================================>   (27 + 2) / 29]

+--------+-----+--------+--------------+------------+--------------+-------------+
|trade_id|price|quantity|quote_quantity|timestamp_ms|is_buyer_maker|is_best_match|
+--------+-----+--------+--------------+------------+--------------+-------------+
|0       |0    |0       |0             |0           |0             |0            |
+--------+-----+--------+--------------+------------+--------------+-------------+



In [10]:
total_rows = df_raw.count()
distinct_ids = df_raw.select("trade_id").distinct().count()
print(f"Total: {total_rows} | Distinct trade_id: {distinct_ids} | Duplicates: {total_rows - distinct_ids}")

df_raw.select(
    F.from_unixtime(F.col("timestamp_ms") / 1000).alias("ts_utc")
).agg(
    F.min("ts_utc").alias("min_ts"),
    F.max("ts_utc").alias("max_ts")
).show(truncate=False)

Total: 52549865 | Distinct trade_id: 52549865 | Duplicates: 0


[Stage 16:================================================>       (25 + 4) / 29]

+-------------------+-------------------+
|min_ts             |max_ts             |
+-------------------+-------------------+
|2024-01-01 02:00:00|2024-02-01 01:59:59|
+-------------------+-------------------+



In [11]:
df_raw.filter(
    (F.col("price") <= 0) | (F.col("quantity") <= 0) | (F.col("quote_quantity") <= 0)
).count()

0